# Embeddings and a Vector Index with FAISS
In this notebook you will:
1. Turn sentences into **embeddings** (vectors of numbers)
2. See that similar meanings give similar vectors
3. Store the vectors in a **FAISS** index (Facebook's fast vector search library)
4. Run a **similarity search**: find the sentences closest in meaning to a question
5. Add more data and learn when to use faster, approximate indexes

## 0. Add your API key
Get a free key at https://aistudio.google.com/apikey, then paste it in the cell below.
**Never share your key or upload it to GitHub.**

In [ ]:
# ==========================================
#  >>> PASTE YOUR API KEY BETWEEN THE QUOTES <<<
# ==========================================
API_KEY = "PASTE_YOUR_API_KEY_HERE"

## 1. Install and set up
We need `google-genai` (to make embeddings) and `faiss-cpu` (the vector database). Run the install cell once.

In [ ]:
%pip install -q google-genai faiss-cpu numpy

In [ ]:
import faiss
import numpy as np
from google import genai
from google.genai import types

if not API_KEY or API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise RuntimeError("Paste your API key into the API_KEY cell above first.")

client = genai.Client(api_key=API_KEY)
EMBED_MODEL = "gemini-embedding-001"
print("Client ready")

## 2. What is an embedding?
An **embedding** turns a piece of text into a list of numbers (a **vector**). Texts with similar *meaning* get vectors that point in similar directions.

Let's embed one sentence and look at it.

In [ ]:
def embed(texts):
    """Turn a list of texts into a list of vectors (768 numbers each)."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    return [e.values for e in result.embeddings]

vec = embed(["The cat sat on the sofa."])[0]
print(f"Numbers in the vector : {len(vec)}")
print(f"First 5 numbers       : {vec[:5]}")

## 3. Closer meaning = closer vectors
**Cosine similarity** measures how close two vectors are: 1.0 means the same direction, and lower means less related. Let's compare a sentence with a similar one and with an unrelated one.

In [ ]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

s1, s2, s3 = embed([
    "The cat sat on the sofa.",
    "A kitten is resting on the couch.",
    "The rocket launched into orbit.",
])

print(f"cat vs kitten : {cosine(s1, s2):.3f}   <- similar meaning, no shared key words")
print(f"cat vs rocket : {cosine(s1, s3):.3f}   <- unrelated, so a lower score")

## 4. Our data
Here is a small knowledge base: 9 sentences about 3 topics (animals, food, space). We type it straight into the notebook and embed all of it in one call.

In [ ]:
# (text, topic) pairs - typed in by hand, no files needed
DATA = [
    ("Dogs are loyal pets that love to play fetch.",           "animals"),
    ("A cheetah can run faster than any other land animal.",   "animals"),
    ("Dolphins are intelligent mammals that live in the sea.", "animals"),
    ("Pizza is made with dough, tomato sauce and cheese.",     "food"),
    ("Sushi is a Japanese dish of rice and raw fish.",         "food"),
    ("A bowl of hot soup is perfect on a cold day.",           "food"),
    ("The Moon orbits the Earth about once a month.",          "space"),
    ("Mars is called the red planet because of its rusty soil.", "space"),
    ("Astronauts float because they are in free fall around Earth.", "space"),
]
DOCUMENTS = [text for text, topic in DATA]
TOPICS    = [topic for text, topic in DATA]

doc_vectors = embed(DOCUMENTS)      # one API call for all sentences
print(f"{len(DOCUMENTS)} sentences -> {len(doc_vectors)} vectors of {len(doc_vectors[0])} numbers each")

## 5. Store the vectors in a FAISS index
FAISS is a library that stores **only vectors**, not the text. So we keep our `DOCUMENTS` list and use the **position** (0, 1, 2, ...) to find the text of each result.

- We **normalize** each vector to length 1. Then the *inner product* (dot product) equals cosine similarity.
- `IndexFlatIP` compares the query with every stored vector (exact search).

In [ ]:
vectors = np.array(doc_vectors, dtype="float32")
faiss.normalize_L2(vectors)                 # make every vector length 1

index = faiss.IndexFlatIP(vectors.shape[1]) # IP = inner product
index.add(vectors)
print(f"Stored {index.ntotal} vectors of {index.d} numbers each")

## 6. Similarity search
We embed the **question** the same way (and normalize it), then ask FAISS for the `K` closest vectors. FAISS returns the **scores** and the **positions** of the matches.

The score is a similarity: bigger = closer in meaning.

In [ ]:
QUERY = "Which animal is the fastest?"
K = 3

def search(query, k=K):
    q = np.array(embed([query]), dtype="float32")
    faiss.normalize_L2(q)
    scores, positions = index.search(q, k)
    print(f"Question: {query}\n")
    for score, pos in zip(scores[0], positions[0]):
        print(f"  similarity {score:.3f} | position {pos} | {DOCUMENTS[pos]}")

search(QUERY)

## 7. Search by meaning, not by key words
The questions below share **no words** with the best answer. A normal text search would fail, but the embeddings still match the meaning.

In [ ]:
search("What do people eat from Japan?")
search("Why is there no gravity in a space station?")

## 8. Add more data
You can keep adding vectors to the index. Remember to add the text to `DOCUMENTS` too, so positions stay in sync.

In [ ]:
NEW_TEXT = "Penguins cannot fly but they are excellent swimmers."
new_vec = np.array(embed([NEW_TEXT]), dtype="float32")
faiss.normalize_L2(new_vec)

index.add(new_vec)
DOCUMENTS.append(NEW_TEXT)
print(f"Now storing {index.ntotal} vectors\n")
search("Which bird swims well?")

## 9. Why FAISS? Speed at scale
`IndexFlatIP` checks **every** vector, which is perfect for 10 sentences but slow for 100 million. FAISS also has *approximate* indexes (for example `IndexIVFFlat` and `IndexHNSWFlat`) that skip most vectors and are much faster, with a tiny loss in accuracy.

Let's time an exact search on 100,000 random vectors to see that it is still fast.

In [ ]:
import time

big = np.random.rand(100_000, 768).astype("float32")
faiss.normalize_L2(big)
big_index = faiss.IndexFlatIP(768)
big_index.add(big)

start = time.time()
big_index.search(big[:1], 3)
print(f"Searched {big_index.ntotal:,} vectors in {(time.time() - start) * 1000:.1f} ms")

## Try it yourself
- Change `QUERY` and `K` in section 6.
- Add 3 sentences of your own (new topic too!) to `DATA` and rerun from section 4.
- Ask a question that matches **nothing** in the data. What does the best score look like?
- Rewrite the same question in different words and check that the same sentence still comes first.